In [ ]:
!pip install datasets evaluate transformers[sentencepiece]
!apt install git-lfs

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 480.6/480.6 kB 35.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.0/84.0 kB 8.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 116.3/116.3 kB 13.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 179.3/179.3 kB 20.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 134.8/134.8 kB 14.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 194.1/194.1 kB 19.2 MB/s eta 0:00:00
  Attempting uninstall: fsspec
    Found existing installation: fsspec 2024.10.0
    Uninstalling fsspec-2024.10.0:
      Successfully uninstalled fsspec-2024.10.0
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
gcsfs 2024.10.0 requires fsspec==2024.10.0, but you have fsspec 2024.9.0 which is incompatible.
Reading package lists... Done
Building dependency tree... Done
Reading state infor

### You will need to setup git, adapt your email and name in the following cell.

In [ ]:
!git config --global user.email "richannan24@gmail.com"
!git config --global user.name "Astonish24"

### You will also need to be logged in to the Hugging Face Hub. Execute the following and enter your credentials.

In [ ]:
from huggingface_hub import notebook_login

notebook_login()

### Dataset

In [ ]:
import numpy as np
import pandas as pd

In [ ]:
import pandas as pd

# Read the CSV file
data = pd.read_csv('sample _questions_for_pilot_test.csv')

# Display the first few rows
data_df = pd.DataFrame(data)
raw_datasets = data_df
raw_datasets.dropna(inplace=True)
raw_datasets.head()

,question,answers,context
0,What is the primary focus of the integrative p...,The primary focus is to integrate sustainable ...,This integrative process seeks to incorporate ...
1,Why is human health prioritized in the integra...,Human health is prioritized to ensure that bui...,This integrative process seeks to incorporate ...
2,At what stage should the cross-discipline appr...,The cross-discipline approach should begin dur...,This integrative process seeks to incorporate ...
3,What document is prepared to establish the fou...,The Owner’s Project Requirements (OPR) document.,This integrative process seeks to incorporate ...
4,What is the purpose of the preliminary LEED me...,What is the purpose of the preliminary LEED mT...,This integrative process seeks to incorporate ...


In [ ]:
from difflib import SequenceMatcher

def find_answer_start_position(answer, context):
    """
    Finds the start position of the answer in the context and returns it in a specific format.

    Parameters:
    answer (str): The answer string to search for.
    context (str): The context string to search in.

    Returns:
    dict: A dictionary in the format {'text': [answer], 'answer_start': [start_index]}.
    """
    match = SequenceMatcher(None, context, answer).find_longest_match(0, len(context), 0, len(answer))
    if match.size > 0:
        start_index = match.a  # Start index in the context
        return {'text': [answer], 'answer_start': [start_index]}
    return {'text': [answer], 'answer_start': [-1]}  # -1 indicates the answer was not found

# # Example usage
# answer = "The primary focus is to integrate sustainable and cost-effective green design and construction strategies with a core emphasis on human health"
# context = """This integrative process seeks to incorporate sustainable and cost-effective green design and construction strategies with a core focus on human health. By embedding health as a primary evaluative criterion in building design, construction, and operational strategies, this approach promotes innovative practices that foster high-performance healing environments for patients, caregivers, and staff."""

# result = find_answer_start_position(answer, context)
# print(result)

In [ ]:
answer_index = []
for i in range(0, raw_datasets.shape[0]):
    answer_index.append(find_answer_start_position(raw_datasets.iloc[i]['answers'],raw_datasets.iloc[i]['question']))
raw_datasets['answer_idx'] = answer_index
# Assuming df is your DataFrame
unique_numbers = [f"unique-{i}" for i in range(raw_datasets.shape[0])]
raw_datasets['id'] = unique_numbers
raw_datasets.to_json()

'{"question":{"0":"What is the primary focus of the integrative process in healthcare projects?","1":"Why is human health prioritized in the integrative process?","2":"At what stage should the cross-discipline approach begin in healthcare projects?","3":"What document is prepared to establish the foundation for a health-focused environment in healthcare projects?","4":"What is the purpose of the preliminary LEED meeting in healthcare projects?","5":"When is it ideal to conduct the preliminary LEED meeting?","6":"How many professionals should the integrated project team include at minimum?","7":"Who might be part of the integrated project team in a healthcare project?","8":"What is the purpose of having a comprehensive project team in healthcare projects?","9":"What collaborative session must be held to enhance planning, and who should attend?","10":"How long should the design charrette session last?","11":"At what project phase is the design charrette ideally conducted?","12":"What is 

In [ ]:
from datasets import Dataset, DatasetDict
import pandas as pd

# Assume raw_datasets is your pandas DataFrame
# Split the dataset into train and validation sets (adjust proportions as needed)
train_df = raw_datasets.sample(frac=0.8, random_state=42)  # 80% for training
validation_df = raw_datasets.drop(train_df.index)  # Remaining 20% for validation

# Convert pandas DataFrames to Hugging Face Datasets
train_dataset = Dataset.from_pandas(train_df)
validation_dataset = Dataset.from_pandas(validation_df)

# Create the DatasetDict
dataset_dict = DatasetDict({
    "train": train_dataset,
    "validation": validation_dataset
})


# Display dataset summary
print(dataset_dict)
raw_datasets = dataset_dict

DatasetDict({
    train: Dataset({
        features: ['question', 'answers', 'context', 'answer_idx', 'id', '__index_level_0__'],
        num_rows: 64
    })
    validation: Dataset({
        features: ['question', 'answers', 'context', 'answer_idx', 'id', '__index_level_0__'],
        num_rows: 16
    })
})


In [ ]:
print(raw_datasets['train'][0:1]['answer_idx'])

[{'answer_start': [128], 'text': ['Avoiding such land ensures the protection of resources essential for food production, ecological health, and water management.']}]


In [ ]:
# from sklearn.model_selection import train_test_split

# # Split the data into training and validation sets
# raw_datasets, raw_datasets_val = train_test_split(
#     raw_datasets,  # The DataFrame containing the processed data
#     test_size=0.2,  # Percentage of the data to be used for validation (e.g., 20%)
#     random_state=42,  # For reproducibility
#     shuffle=True  # Shuffle the data before splitting
# )

# # Split the validation set into validation and test sets
# raw_datasets_val, raw_datasets_test = train_test_split(
#     raw_datasets_val,  # The previously created validation DataFrame
#     test_size=0.1,  # Split 50% of the validation set into test data
#     random_state=42,  # For reproducibility
#     shuffle=True  # Shuffle the data before splitting
# )

# # Display the sizes of the splits
# print(f"Training set size: {len(raw_datasets)}")
# print(f"Validation set size: {len(raw_datasets_val)}")
# print(f"Test set size: {len(raw_datasets_test)}")

In [ ]:
print("Context: ", raw_datasets["train"][0]["context"])
print("Question: ", raw_datasets["train"][0]["question"])
print("Answer: ", raw_datasets["train"][0]["answer_idx"])

Context:  The Sensitive Land Protection credit is another important component of the LT credits, awarding 1 to 2 points to projects that avoid environmentally sensitive lands, thus minimizing the ecological impact of development. The credit offers two options: locating on previously developed land or choosing land that does not meet criteria for sensitivity, such as prime farmland, floodplains, or habitats for endangered species. This protection extends to water bodies and wetlands, requiring a minimum buffer of 100 feet for water bodies and 50 feet for wetlands. Projects aiming for this credit must be careful to avoid land identified as critical for agriculture, ecosystems, or water retention. Sensitive habitats are defined by regulations like the U.S. Endangered Species Act, NatureServe classifications, or equivalent local standards, ensuring protection for vulnerable ecosystems and species. For projects on previously developed land or those that meet sensitive land standards, minor 

In [ ]:
raw_datasets["train"].filter(lambda x: len(x["answer_idx"]["text"]) != 1)

Filter:   0%|          | 0/64 [00:00<?, ? examples/s]

Dataset({
    features: ['question', 'answers', 'context', 'answer_idx', 'id', '__index_level_0__'],
    num_rows: 0
})

In [ ]:
print(raw_datasets["validation"][0]["answer_idx"])
print(raw_datasets["validation"][2]["answer_idx"])

{'answer_start': [19], 'text': ['Human health is prioritized to ensure that building design, construction, and operational strategies foster high-performance healing environments for patients, caregivers, and staff.']}
{'answer_start': [16], 'text': ['The primary purpose of the Integrative Process credit is to enhance high-performance and cost-effective project outcomes by analyzing and leveraging the interrelationships among various building systems early in the design process.']}


In [ ]:
print(raw_datasets["validation"][2]["context"])
print(raw_datasets["validation"][2]["question"])

The Integrative Process credit for BD&C projects aims to enhance high-performance, cost-effective outcomes by analyzing and leveraging the interrelationships among different building systems. This credit applies across various project types, including new construction, core and shell, schools, retail, data centers, warehouses, distribution centers, hospitality, and healthcare facilities. The process begins during the pre-design phase and continues through the design stages to maximize synergy across disciplines and building systems. By incorporating these interdisciplinary insights into essential project documents, such as the Owner’s Project Requirements (OPR) and Basis of Design (BOD), teams can ensure the resulting design aligns with both operational efficiency and sustainability goals.
What is the main purpose of the Integrative Process credit in BD&C projects?


In [ ]:
from transformers import AutoTokenizer

model_checkpoint = "bert-base-cased"
tokenizer = AutoTokenizer.from_pretrained(model_checkpoint)

/usr/local/lib/python3.10/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json:   0%|          | 0.00/49.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/213k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/436k [00:00<?, ?B/s]

In [ ]:
tokenizer.is_fast

True

In [ ]:
context = raw_datasets["train"][0]["context"]
question = raw_datasets["train"][0]["question"]

inputs = tokenizer(question, context)
tokenizer.decode(inputs["input_ids"])

'[CLS] Why is it important to avoid land identified as critical for agriculture, ecosystems, or water retention in the Sensitive Land Protection credit? [SEP] The Sensitive Land Protection credit is another important component of the LT credits, awarding 1 to 2 points to projects that avoid environmentally sensitive lands, thus minimizing the ecological impact of development. The credit offers two options : locating on previously developed land or choosing land that does not meet criteria for sensitivity, such as prime farmland, floodplains, or habitats for endangered species. This protection extends to water bodies and wetlands, requiring a minimum buffer of 100 feet for water bodies and 50 feet for wetlands. Projects aiming for this credit must be careful to avoid land identified as critical for agriculture, ecosystems, or water retention. Sensitive habitats are defined by regulations like the U. S. Endangered Species Act, NatureServe classifications, or equivalent local standards, e

In [ ]:
inputs = tokenizer(
    question,
    context,
    max_length=100,
    truncation="only_second",
    stride=50,
    return_overflowing_tokens=True,
)

for ids in inputs["input_ids"]:
    print(tokenizer.decode(ids))

[CLS] Why is it important to avoid land identified as critical for agriculture, ecosystems, or water retention in the Sensitive Land Protection credit? [SEP] The Sensitive Land Protection credit is another important component of the LT credits, awarding 1 to 2 points to projects that avoid environmentally sensitive lands, thus minimizing the ecological impact of development. The credit offers two options : locating on previously developed land or choosing land that does not meet criteria for sensitivity, such as prime farmland, floodplains, [SEP]
[CLS] Why is it important to avoid land identified as critical for agriculture, ecosystems, or water retention in the Sensitive Land Protection credit? [SEP] to projects that avoid environmentally sensitive lands, thus minimizing the ecological impact of development. The credit offers two options : locating on previously developed land or choosing land that does not meet criteria for sensitivity, such as prime farmland, floodplains, or habitat

In [ ]:
inputs = tokenizer(
    question,
    context,
    max_length=100,
    truncation="only_second",
    stride=50,
    return_overflowing_tokens=True,
)

for ids in inputs["input_ids"]:
    print(tokenizer.decode(ids))

[CLS] Why is it important to avoid land identified as critical for agriculture, ecosystems, or water retention in the Sensitive Land Protection credit? [SEP] The Sensitive Land Protection credit is another important component of the LT credits, awarding 1 to 2 points to projects that avoid environmentally sensitive lands, thus minimizing the ecological impact of development. The credit offers two options : locating on previously developed land or choosing land that does not meet criteria for sensitivity, such as prime farmland, floodplains, [SEP]
[CLS] Why is it important to avoid land identified as critical for agriculture, ecosystems, or water retention in the Sensitive Land Protection credit? [SEP] to projects that avoid environmentally sensitive lands, thus minimizing the ecological impact of development. The credit offers two options : locating on previously developed land or choosing land that does not meet criteria for sensitivity, such as prime farmland, floodplains, or habitat

In [ ]:
inputs = tokenizer(
    question,
    context,
    max_length=100,
    truncation="only_second",
    stride=50,
    return_overflowing_tokens=True,
    return_offsets_mapping=True,
)
inputs.keys()

dict_keys(['input_ids', 'token_type_ids', 'attention_mask', 'offset_mapping', 'overflow_to_sample_mapping'])

In [ ]:
inputs["overflow_to_sample_mapping"]

[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0]

In [ ]:
inputs = tokenizer(
    raw_datasets["train"][2:6]["question"],
    raw_datasets["train"][2:6]["context"],
    max_length=100,
    truncation="only_second",
    stride=50,
    return_overflowing_tokens=True,
    return_offsets_mapping=True,
)

print(f"The 4 examples gave {len(inputs['input_ids'])} features.")
print(f"Here is where each comes from: {inputs['overflow_to_sample_mapping']}.")

The 4 examples gave 17 features.
Here is where each comes from: [0, 0, 0, 0, 0, 1, 1, 1, 1, 2, 2, 2, 2, 3, 3, 3, 3].


In [ ]:
answers = raw_datasets["train"][2:6]["answer_idx"]
print(answers)

[{'answer_start': [3], 'text': ['By incorporating multifunctional spaces, projects can reduce the overall building area and optimize space usage, thereby lowering energy and water demands and creating a more efficient, adaptable design.']}, {'answer_start': [6], 'text': ['Minor improvements such as narrow bicycle paths, native plant restoration, and small clearings are allowed to support sustainable access and appreciation of these natural areas.']}, {'answer_start': [1], 'text': ['The water budget analysis is crucial for identifying methods to reduce potable water use and exploring nonpotable sources, optimizing systems like plumbing, rainwater management, and landscaping for greater water efficiency.']}, {'answer_start': [56], 'text': ['By awarding points to projects that avoid environmentally sensitive lands, the credit minimizes the ecological impact of development.']}]


In [ ]:
answers = raw_datasets["train"][2:6]["answer_idx"]
start_positions = []
end_positions = []

for i, offset in enumerate(inputs["offset_mapping"]):
    sample_idx = inputs["overflow_to_sample_mapping"][i]
    answer = answers[sample_idx]
    start_char = answer["answer_start"][0]
    end_char = answer["answer_start"][0] + len(answer["text"][0])
    sequence_ids = inputs.sequence_ids(i)

    # Find the start and end of the context
    idx = 0
    while sequence_ids[idx] != 1:
        idx += 1
    context_start = idx
    while sequence_ids[idx] == 1:
        idx += 1
    context_end = idx - 1

    # If the answer is not fully inside the context, label is (0, 0)
    if offset[context_start][0] > start_char or offset[context_end][1] < end_char:
        start_positions.append(0)
        end_positions.append(0)
    else:
        # Otherwise it's the start and end token positions
        idx = context_start
        while idx <= context_end and offset[idx][0] <= start_char:
            idx += 1
        start_positions.append(idx - 1)

        idx = context_end
        while idx >= context_start and offset[idx][1] >= end_char:
            idx -= 1
        end_positions.append(idx + 1)

start_positions, end_positions

([19, 0, 0, 0, 0, 19, 0, 0, 0, 18, 0, 0, 0, 23, 0, 0, 0],
 [60, 0, 0, 0, 0, 46, 0, 0, 0, 59, 0, 0, 0, 49, 0, 0, 0])

In [ ]:
idx = 0
sample_idx = inputs["overflow_to_sample_mapping"][idx]
answer = answers[sample_idx]["text"][0]

start = start_positions[idx]
end = end_positions[idx]
labeled_answer = tokenizer.decode(inputs["input_ids"][idx][start : end + 1])

print(f"Theoretical answer: {answer}, labels give: {labeled_answer}")

Theoretical answer: By incorporating multifunctional spaces, projects can reduce the overall building area and optimize space usage, thereby lowering energy and water demands and creating a more efficient, adaptable design., labels give: Energy - related systems are a core focus within the Integrative Process, requiring a preliminary energy modeling analysis, often called a “ simple box ” model, completed before the schematic design phase. This


In [ ]:
idx = 4
sample_idx = inputs["overflow_to_sample_mapping"][idx]
answer = answers[sample_idx]["text"][0]

decoded_example = tokenizer.decode(inputs["input_ids"][idx])
print(f"Theoretical answer: {answer}, decoded example: {decoded_example}")

Theoretical answer: By incorporating multifunctional spaces, projects can reduce the overall building area and optimize space usage, thereby lowering energy and water demands and creating a more efficient, adaptable design., decoded example: [CLS] In what ways might multifunctional spaces contribute to the Integrative Process? [SEP], thermal comfort ranges, and plug load demands are analyzed to identify opportunities for energy efficiency. Documentation of how these analyses inform decisions is then reflected in the project ’ s form, geometry, and building envelope features, often resulting in significant downsizing or simplification of building systems such as HVAC, lighting, and other controls. [SEP]


In [ ]:
max_length = 384
stride = 128


def preprocess_training_examples(examples):
    questions = [q.strip() for q in examples["question"]]
    inputs = tokenizer(
        questions,
        examples["context"],
        max_length=max_length,
        truncation="only_second",
        stride=stride,
        return_overflowing_tokens=True,
        return_offsets_mapping=True,
        padding="max_length",
    )

    offset_mapping = inputs.pop("offset_mapping")
    sample_map = inputs.pop("overflow_to_sample_mapping")
    answers = examples["answer_idx"]
    start_positions = []
    end_positions = []

    for i, offset in enumerate(offset_mapping):
        sample_idx = sample_map[i]
        answer = answers[sample_idx]
        start_char = answer["answer_start"][0]
        end_char = answer["answer_start"][0] + len(answer["text"][0])
        sequence_ids = inputs.sequence_ids(i)

        # Find the start and end of the context
        idx = 0
        while sequence_ids[idx] != 1:
            idx += 1
        context_start = idx
        while sequence_ids[idx] == 1:
            idx += 1
        context_end = idx - 1

        # If the answer is not fully inside the context, label is (0, 0)
        if offset[context_start][0] > start_char or offset[context_end][1] < end_char:
            start_positions.append(0)
            end_positions.append(0)
        else:
            # Otherwise it's the start and end token positions
            idx = context_start
            while idx <= context_end and offset[idx][0] <= start_char:
                idx += 1
            start_positions.append(idx - 1)

            idx = context_end
            while idx >= context_start and offset[idx][1] >= end_char:
                idx -= 1
            end_positions.append(idx + 1)

    inputs["start_positions"] = start_positions
    inputs["end_positions"] = end_positions
    return inputs

In [ ]:
train_dataset = raw_datasets["train"].map(
    preprocess_training_examples,
    batched=True,
    remove_columns=raw_datasets["train"].column_names,
)
len(raw_datasets["train"]), len(train_dataset)

Map:   0%|          | 0/64 [00:00<?, ? examples/s]

(64, 64)

In [ ]:
print(train_dataset[0])

{'input_ids': [101, 2009, 1110, 1122, 1696, 1106, 3644, 1657, 3626, 1112, 3607, 1111, 6487, 117, 23600, 117, 1137, 1447, 23406, 1107, 1103, 14895, 22472, 4026, 8063, 4755, 136, 102, 1109, 14895, 22472, 4026, 8063, 4755, 1110, 1330, 1696, 6552, 1104, 1103, 149, 1942, 6459, 117, 25055, 122, 1106, 123, 1827, 1106, 3203, 1115, 3644, 4801, 1193, 7246, 4508, 117, 2456, 8715, 25596, 1103, 14769, 3772, 1104, 1718, 119, 1109, 4755, 3272, 1160, 6665, 131, 25338, 14829, 1113, 2331, 1872, 1657, 1137, 11027, 1657, 1115, 1674, 1136, 2283, 9173, 1111, 15750, 117, 1216, 1112, 5748, 17790, 117, 7870, 18220, 1116, 117, 1137, 10902, 1111, 11532, 1530, 119, 1188, 3636, 8559, 1106, 1447, 3470, 1105, 20432, 117, 8753, 170, 5867, 20232, 1104, 1620, 1623, 1111, 1447, 3470, 1105, 1851, 1623, 1111, 20432, 119, 21454, 14485, 1111, 1142, 4755, 1538, 1129, 5784, 1106, 3644, 1657, 3626, 1112, 3607, 1111, 6487, 117, 23600, 117, 1137, 1447, 23406, 119, 14895, 22472, 10902, 1132, 3393, 1118, 7225, 1176, 1103, 158, 119

In [ ]:
def preprocess_validation_examples(examples):
    questions = [q.strip() for q in examples["question"]]
    inputs = tokenizer(
        questions,
        examples["context"],
        max_length=max_length,
        truncation="only_second",
        stride=stride,
        return_overflowing_tokens=True,
        return_offsets_mapping=True,
        padding="max_length",
    )

    sample_map = inputs.pop("overflow_to_sample_mapping")
    example_ids = []

    for i in range(len(inputs["input_ids"])):
        sample_idx = sample_map[i]
        example_ids.append(examples["id"][sample_idx])

        sequence_ids = inputs.sequence_ids(i)
        offset = inputs["offset_mapping"][i]
        inputs["offset_mapping"][i] = [
            o if sequence_ids[k] == 1 else None for k, o in enumerate(offset)
        ]

    inputs["example_id"] = example_ids
    return inputs
# example_ids = []
# def preprocess_validation_examples(examples, unique_id_start=1):
#     """
#     Preprocess validation examples and generate unique IDs for each example.

#     Parameters:
#     - examples: The dataset examples containing 'question', 'context', etc.
#     - unique_id_start: Starting value for generating unique IDs.

#     Returns:
#     - Tokenized inputs with generated unique example IDs.
#     """
#     questions = [q.strip() for q in examples["question"]]
#     inputs = tokenizer(
#         questions,
#         examples["context"],
#         max_length=max_length,
#         truncation="only_second",
#         stride=stride,
#         return_overflowing_tokens=True,
#         return_offsets_mapping=True,
#         padding="max_length",
#     )

#     sample_map = inputs.pop("overflow_to_sample_mapping")
#     #global example_ids

#     # Counter for unique ID generation
#     current_id = unique_id_start

#     for i in range(len(inputs["input_ids"])):
#         # Generate a unique ID
#         unique_id = f"unique-{current_id}"
#         current_id += 1
#         example_ids.append(unique_id)  # Append the generated unique ID

#         sequence_ids = inputs.sequence_ids(i)
#         offset = inputs["offset_mapping"][i]
#         inputs["offset_mapping"][i] = [
#             o if sequence_ids[k] == 1 else None for k, o in enumerate(offset)
#         ]

#     inputs["example_id"] = example_ids
#     return inputs

In [ ]:
validation_dataset = raw_datasets["validation"].map(
    preprocess_validation_examples,
    batched=True,
    remove_columns=raw_datasets["validation"].column_names,
)
len(raw_datasets["validation"]), len(validation_dataset)

Map:   0%|          | 0/16 [00:00<?, ? examples/s]

(16, 17)

In [ ]:
validation_dataset

Dataset({
    features: ['input_ids', 'token_type_ids', 'attention_mask', 'offset_mapping', 'example_id'],
    num_rows: 17
})

In [ ]:
small_eval_set = raw_datasets["validation"].select(range(4))
trained_checkpoint = "distilbert-base-cased-distilled-squad"

tokenizer = AutoTokenizer.from_pretrained(trained_checkpoint)
eval_set = small_eval_set.map(
    preprocess_validation_examples,
    batched=True,
    remove_columns=raw_datasets["validation"].column_names,
)

tokenizer_config.json:   0%|          | 0.00/49.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/473 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/213k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/436k [00:00<?, ?B/s]

Map:   0%|          | 0/4 [00:00<?, ? examples/s]

In [ ]:
eval_set

Dataset({
    features: ['input_ids', 'attention_mask', 'offset_mapping', 'example_id'],
    num_rows: 4
})

In [ ]:
tokenizer = AutoTokenizer.from_pretrained(model_checkpoint)

In [ ]:
import tensorflow as tf
from transformers import TFAutoModelForQuestionAnswering

eval_set_for_model = eval_set.remove_columns(["example_id", "offset_mapping"])
eval_set_for_model.set_format("numpy")

batch = {k: eval_set_for_model[k] for k in eval_set_for_model.column_names}
trained_model = TFAutoModelForQuestionAnswering.from_pretrained(trained_checkpoint)

outputs = trained_model(**batch)

model.safetensors:   0%|          | 0.00/261M [00:00<?, ?B/s]

All PyTorch model weights were used when initializing TFDistilBertForQuestionAnswering.

All the weights of TFDistilBertForQuestionAnswering were initialized from the PyTorch model.
If your task is similar to the task the model of the checkpoint was trained on, you can already use TFDistilBertForQuestionAnswering for predictions without further training.


In [ ]:
start_logits = outputs.start_logits.numpy()
end_logits = outputs.end_logits.numpy()

In [ ]:
import collections

example_to_features = collections.defaultdict(list)
for idx, feature in enumerate(eval_set):
    example_to_features[feature["example_id"]].append(idx)

In [ ]:
small_eval_set

Dataset({
    features: ['question', 'answers', 'context', 'answer_idx', 'id', '__index_level_0__'],
    num_rows: 4
})

In [ ]:
import numpy as np

n_best = 20
max_answer_length = 30
predicted_answers = []

for example in small_eval_set:
    example_id = example["id"]
    context = example["context"]
    answers = []

    for feature_index in example_to_features[example_id]:
        start_logit = start_logits[feature_index]
        end_logit = end_logits[feature_index]
        offsets = eval_set["offset_mapping"][feature_index]

        start_indexes = np.argsort(start_logit)[-1 : -n_best - 1 : -1].tolist()
        end_indexes = np.argsort(end_logit)[-1 : -n_best - 1 : -1].tolist()
        for start_index in start_indexes:
            for end_index in end_indexes:
                # Skip answers that are not fully in the context
                if offsets[start_index] is None or offsets[end_index] is None:
                    continue
                # Skip answers with a length that is either < 0 or > max_answer_length.
                if (
                    end_index < start_index
                    or end_index - start_index + 1 > max_answer_length
                ):
                    continue

                answers.append(
                    {
                        "text": context[offsets[start_index][0] : offsets[end_index][1]],
                        "logit_score": start_logit[start_index] + end_logit[end_index],
                    }
                )

    best_answer = max(answers, key=lambda x: x["logit_score"])
    predicted_answers.append({"id": example_id, "prediction_text": best_answer["text"]})

In [ ]:
import evaluate

metric = evaluate.load("squad")

In [ ]:
theoretical_answers = [
    {"id": ex["id"], "answers": ex["answer_idx"]} for ex in small_eval_set
]

In [ ]:
print(predicted_answers[0])
print(theoretical_answers[0])

{'id': 'unique-1', 'prediction_text': 'primary evaluative criterion in building design, construction, and operational strategies'}
{'id': 'unique-1', 'answers': {'answer_start': [19], 'text': ['Human health is prioritized to ensure that building design, construction, and operational strategies foster high-performance healing environments for patients, caregivers, and staff.']}}


In [ ]:
metric.compute(predictions=predicted_answers, references=theoretical_answers)

{'exact_match': 0.0, 'f1': 25.803571428571427}

In [ ]:
from tqdm.auto import tqdm


def compute_metrics(start_logits, end_logits, features, examples):
    example_to_features = collections.defaultdict(list)
    for idx, feature in enumerate(features):
        example_to_features[feature["example_id"]].append(idx)

    predicted_answers = []
    for example in tqdm(examples):
        example_id = example["id"]
        context = example["context"]
        answers = []

        # Loop through all features associated with that example
        for feature_index in example_to_features[example_id]:
            start_logit = start_logits[feature_index]
            end_logit = end_logits[feature_index]
            offsets = features[feature_index]["offset_mapping"]

            start_indexes = np.argsort(start_logit)[-1 : -n_best - 1 : -1].tolist()
            end_indexes = np.argsort(end_logit)[-1 : -n_best - 1 : -1].tolist()
            for start_index in start_indexes:
                for end_index in end_indexes:
                    # Skip answers that are not fully in the context
                    if offsets[start_index] is None or offsets[end_index] is None:
                        continue
                    # Skip answers with a length that is either < 0 or > max_answer_length
                    if (
                        end_index < start_index
                        or end_index - start_index + 1 > max_answer_length
                    ):
                        continue

                    answer = {
                        "text": context[offsets[start_index][0] : offsets[end_index][1]],
                        "logit_score": start_logit[start_index] + end_logit[end_index],
                    }
                    answers.append(answer)

        # Select the answer with the best score
        if len(answers) > 0:
            best_answer = max(answers, key=lambda x: x["logit_score"])
            predicted_answers.append(
                {"id": example_id, "prediction_text": best_answer["text"]}
            )
        else:
            predicted_answers.append({"id": example_id, "prediction_text": ""})

    theoretical_answers = [{"id": ex["id"], "answers": ex["answer_idx"]} for ex in examples]
    return metric.compute(predictions=predicted_answers, references=theoretical_answers)

In [ ]:
compute_metrics(start_logits, end_logits, eval_set, small_eval_set)

  0%|          | 0/4 [00:00<?, ?it/s]

{'exact_match': 0.0, 'f1': 25.803571428571427}

In [ ]:
model = TFAutoModelForQuestionAnswering.from_pretrained(model_checkpoint)

All model checkpoint layers were used when initializing TFBertForQuestionAnswering.

All the layers of TFBertForQuestionAnswering were initialized from the model checkpoint at Astonish24/bert-finetuned-squad.
If your task is similar to the task the model of the checkpoint was trained on, you can already use TFBertForQuestionAnswering for predictions without further training.


In [ ]:
from huggingface_hub import notebook_login

notebook_login()

In [ ]:
from transformers import DefaultDataCollator

data_collator = DefaultDataCollator(return_tensors="tf")

In [ ]:
tf_train_dataset = model.prepare_tf_dataset(
    train_dataset,
    collate_fn=data_collator,
    shuffle=True,
    batch_size=16,
)
tf_eval_dataset = model.prepare_tf_dataset(
    validation_dataset,
    collate_fn=data_collator,
    shuffle=False,
    batch_size=16,
)

In [ ]:
from transformers import create_optimizer
from transformers.keras_callbacks import PushToHubCallback
import tensorflow as tf

# The number of training steps is the number of samples in the dataset, divided by the batch size then multiplied
# by the total number of epochs. Note that the tf_train_dataset here is a batched tf.data.Dataset,
# not the original Hugging Face Dataset, so its len() is already num_samples // batch_size.
num_train_epochs = 20
num_train_steps = len(tf_train_dataset) * num_train_epochs
optimizer, schedule = create_optimizer(
    init_lr=2e-5,
    num_warmup_steps=0,
    num_train_steps=num_train_steps,
    weight_decay_rate=0.01,
)
model.compile(optimizer=optimizer)

# Train in mixed-precision float16
tf.keras.mixed_precision.set_global_policy("mixed_float16")

In [ ]:
from transformers.keras_callbacks import PushToHubCallback

callback = PushToHubCallback(output_dir="bert-finetuned-squad", tokenizer=tokenizer)

# We're going to do validation afterwards, so no validation mid-training
model.fit(tf_train_dataset, callbacks=[callback], epochs=num_train_epochs)

/usr/local/lib/python3.10/dist-packages/huggingface_hub/utils/_deprecation.py:131: FutureWarning: 'Repository' (from 'huggingface_hub.repository') is deprecated and will be removed from version '1.0'. Please prefer the http-based alternatives instead. Given its large adoption in legacy code, the complete removal is only planned on next major release.
For more details, please read https://huggingface.co/docs/huggingface_hub/concepts/git_vs_http.
  warnings.warn(warning_message, FutureWarning)
/content/bert-finetuned-squad is already a clone of https://huggingface.co/Astonish24/bert-finetuned-squad. Make sure you pull the latest changes with `repo.git_pull()`.


Epoch 1/20
4/4 [==============================] - 34s 6s/step - loss: 3.6176e-04
Epoch 2/20
4/4 [==============================] - ETA: 0s - loss: 3.1605e-04

Several commits (2) will be pushed upstream.


4/4 [==============================] - 20s 7s/step - loss: 3.1605e-04
Epoch 3/20
4/4 [==============================] - ETA: 0s - loss: 0.0010

Several commits (3) will be pushed upstream.


...

4/4 [==============================] - 20s 7s/step - loss: 5.5569e-04
Epoch 19/20
4/4 [==============================] - ETA: 0s - loss: 3.8087e-04

Several commits (19) will be pushed upstream.


4/4 [==============================] - 20s 7s/step - loss: 3.8087e-04
Epoch 20/20
4/4 [==============================] - ETA: 0s - loss: 5.7936e-04

Several commits (20) will be pushed upstream.


4/4 [==============================] - 20s 6s/step - loss: 5.7936e-04


In [ ]:
predictions = model.predict(tf_eval_dataset)
compute_metrics(
    predictions["start_logits"],
    predictions["end_logits"],
    validation_dataset,
    raw_datasets["validation"],
)

2/2 [==============================] - 3s 25ms/step


  0%|          | 0/16 [00:00<?, ?it/s]

{'exact_match': 0.0, 'f1': 15.462085947510554}

In [ ]:
from transformers import pipeline

# Replace this with your own checkpoint
model_checkpoint = "Astonish24/bert-finetuned-squad"#"huggingface-course/bert-finetuned-squad"
question_answerer = pipeline("question-answering", model=model_checkpoint)

context = """
An uncovered deck is a flat, roofless platform, typically elevated and adjoining a house,
usually made of lumber or similar materials and enclosed by a railing. Density credit
refers to the potential development capacity of a parcel of land, expressed as dwelling
units or other measures of density or intensity. This credit can be transferred to other
parts of the same parcel or contiguous land under a common development plan.
"""
question = "What materials are typically used to construct an uncovered deck?"
question_answerer(question=question, context=context)

All model checkpoint layers were used when initializing TFBertForQuestionAnswering.

All the layers of TFBertForQuestionAnswering were initialized from the model checkpoint at Astonish24/bert-finetuned-squad.
If your task is similar to the task the model of the checkpoint was trained on, you can already use TFBertForQuestionAnswering for predictions without further training.
Device set to use 0


{'score': 1.0,
 'start': 50,
 'end': 125,
 'answer': 'typically elevated and adjoining a house, usually made of lumber or similar'}

### **Using The Trained Model**



In [ ]:
# Use a pipeline as a high-level helper
from transformers import pipeline

pipe = pipeline("question-answering", model="Astonish24/bert-finetuned-squad")

/usr/local/lib/python3.10/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/659 [00:00<?, ?B/s]

tf_model.h5:   0%|          | 0.00/431M [00:00<?, ?B/s]

All model checkpoint layers were used when initializing TFBertForQuestionAnswering.

All the layers of TFBertForQuestionAnswering were initialized from the model checkpoint at Astonish24/bert-finetuned-squad.
If your task is similar to the task the model of the checkpoint was trained on, you can already use TFBertForQuestionAnswering for predictions without further training.


tokenizer_config.json:   0%|          | 0.00/1.22k [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/213k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/669k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/125 [00:00<?, ?B/s]

Device set to use 0


In [ ]:
context = """
An uncovered deck is a flat, roofless platform, typically elevated and adjoining a house,
usually made of lumber or similar materials and enclosed by a railing. Density credit
refers to the potential development capacity of a parcel of land, expressed as dwelling
units or other measures of density or intensity. This credit can be transferred to other
parts of the same parcel or contiguous land under a common development plan.
"""
question = "What materials are typically used to construct an uncovered deck?"
pipe(question=question, context=context)

{'score': 0.9999620914459229,
 'start': 50,
 'end': 126,
 'answer': 'typically elevated and adjoining a house, \nusually made of lumber or similar'}

In [ ]:
context = """
A detached building or structure is one that is not structurally attached or integrated
with another building or structure. Development includes any manmade alterations to real
estate, such as constructing buildings, mining, grading, or paving. Development activity
requiring a floodplain development permit encompasses any such changes, including non-structural
measures like erosion control, that fall under Flood Damage Prevention regulations.
"""
question = "What type of permit is required for development activities in a floodplain?"
pipe(question=question, context=context)

{'score': 6.466458216891624e-06, 'start': 181, 'end': 187, 'answer': 'estate'}

In [ ]:
# # Load model directly
# from transformers import AutoTokenizer, TFAutoModelForQuestionAnswering

# tokenizer = AutoTokenizer.from_pretrained("Astonish24/bert-finetuned-squad")
# # Removed from_tf=True as it is not a TensorFlow model
# model = TFAutoModelForQuestionAnswering.from_pretrained("Astonish24/bert-finetuned-squad")

All model checkpoint layers were used when initializing TFBertForQuestionAnswering.

All the layers of TFBertForQuestionAnswering were initialized from the model checkpoint at Astonish24/bert-finetuned-squad.
If your task is similar to the task the model of the checkpoint was trained on, you can already use TFBertForQuestionAnswering for predictions without further training.
